##Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

##Load Dataset

In [2]:
df = pd.read_csv("cleaned_fw_dataset.csv")
df.head()

,Product_Name,Category,Date_Received,Expiration_Date,Stock_Quantity,Unit_Price,Sales_Volume,Inventory_Turnover_Rate,Shelf_Life,Food_Waste_kg
0,Bell Pepper,Fruits & Vegetables,2024-03-01,2025-01-31,46,4.6,96,55,336.0,0.00000
1,Vegetable Oil,Oils & Fats,2024-04-01,2024-06-11,51,2.0,24,83,71.0,0.37500
2,Parmesan Cheese,Dairy,2024-04-01,2024-04-08,38,12.0,35,24,7.0,0.37500
3,Carrot,Fruits & Vegetables,2024-05-01,2024-09-26,51,1.5,44,95,148.0,0.04698
4,Garlic,Fruits & Vegetables,2024-05-01,2024-05-20,27,7.0,91,77,19.0,0.00000


##Feature Engineering

In [3]:
df['Date_Received'] = pd.to_datetime(df['Date_Received'])
df['Expiration_Date'] = pd.to_datetime(df['Expiration_Date'])

df['days_to_expire'] = (df['Expiration_Date'] - df['Date_Received']).dt.days

Drop unnecessary columns:

In [4]:
df = df.drop(['Product_Name','Date_Received','Expiration_Date'], axis=1)

##Define Features & Target

In [5]:
X = df.drop("Food_Waste_kg", axis=1)
y = df["Food_Waste_kg"]

##Handle Categorical Data

In [6]:
categorical_cols = ['Category']
numeric_cols = X.select_dtypes(include=np.number).columns

Preprocessing pipeline:

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', LabelEncoder(), 'Category'),
        ('num', StandardScaler(), numeric_cols)
    ],
    remainder='passthrough'
)

Instead of LabelEncoder inside pipeline, better do:

In [8]:
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ])

##Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

##Build Model Comparison Pipeline

In [10]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor()
}

##Train & Compare Algorithms

In [11]:
results = []

for name, model in models.items():

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)

    results.append([name, r2, mae])

results_df = pd.DataFrame(results, columns=["Model","R2 Score","MAE"])
results_df.sort_values(by="R2 Score", ascending=False)

,Model,R2 Score,MAE
2,Random Forest,0.970422,0.900677
1,Decision Tree,0.962164,1.088491
4,KNN,0.774339,3.499770
0,Linear Regression,0.464932,8.329464
3,SVR,0.381013,5.968880


##Find Best Model

In [12]:
best_model = results_df.sort_values(by="R2 Score", ascending=False).iloc[0]
best_model

,2
Model,Random Forest
R2 Score,0.970422
MAE,0.900677
